In [1]:
import requests
import os
from dotenv import load_dotenv
import  logging 


load_dotenv()

username = os.getenv("MSP_USERNAME")
password = os.getenv("MSP_PASSWORD")
scope = os.getenv("MSP_SCOPE")
start_url = "https://msp.adionsystems.com/api/beginsession"

headers = {
    'Content-Type': 'application/json'
    }

def get_token():
    logging.info("Deriving session token")
    
    payload = {
    'username': username,
    'password': password,
    'scope': scope
    }
    try:
        res = requests.request("POST", start_url, headers=headers, json=payload, timeout=None)
        res.raise_for_status()
        if res.status_code == 200:
            token = res.json()['authorizationResult'].get('token')
            logging.info("succesfully derived session token")
            return token
        
    except Exception as e:
        logging.warning(f'Bad request check parameters {e}')


In [2]:
import pandas as pd 
import requests
def fetch_po(token):
    url = f"https://msp.adionsystems.com/api/graphql?token={token}"
    headers = {"content-type": "application/json"}

    page_start = 0
    page_size = 1000
    all_records = []

    query = """
    query customerPOs($pageSize: Int, $pageStart: Int, $query: CustomerPOQuery) {
        customerPOs(pageSize: $pageSize, pageStart: $pageStart, query: $query) {
            records {
                poId
                createdTime
                lastModifiedTime
                clientPONumber
                clientPlainText
                client {
                    id
                    name
                    code
                }
                totalAmount
                status
                dueDate
                estimatedStartDate
                description
                internalNotes
                contact
                shippingAddress {
                    line1
                    line2
                    city
                    state
                    zipCode
                    country
                }
                billingAddress {
                    line1
                    line2
                    city
                    state
                    zipCode
                    country
                }
                lineItems {
                    id
                    partNumber
                    partName
                    quantity
                    unitPrice
                    totalPrice
                    description
                }
            }
            totalRecords
        }
    }
    """

    while True:
        payload = {
            "query": query,
            "variables": {
                "pageSize": page_size,
                "pageStart": page_start,
                "query": {
                    "lastModifiedTime": {
                        "greaterThanOrEqual": "2024-01-01T00:00:00Z"
                    }
                }
            }
        }

        res = requests.post(url, headers=headers, json=payload)
        res.raise_for_status()

        data = res.json()["data"]["customerPOs"]
        records = data.get("records", [])
        total_records = data.get("totalRecords", 0)

        logging.info(f"Fetched {len(records)} records (pageStart={page_start})")

        if not records:
            break

        all_records.extend(records)
        page_start += page_size

        if page_start >= total_records:
            break

    logging.info(f"Total fetched records: {len(all_records)}")
    return all_records

def parse_data(records):
    """Parse data received from API"""  
    parsed_records = []
    for rec in records:
        # Extract client information
        client = rec.get("client", {})
        client_name = client.get("name") if client else None
        client_id = client.get("id") if client else None
        client_code = client.get("code") if client else None
        
        # Extract shipping address
        shipping = rec.get("shippingAddress", {})
        shipping_addr = None
        if shipping:
            shipping_addr = f"{shipping.get('line1', '')} {shipping.get('line2', '')}".strip()
            shipping_city = shipping.get('city')
            shipping_state = shipping.get('state')
            shipping_zip = shipping.get('zipCode')
        
        # Extract billing address
        billing = rec.get("billingAddress", {})
        billing_addr = None
        if billing:
            billing_addr = f"{billing.get('line1', '')} {billing.get('line2', '')}".strip()
            billing_city = billing.get('city')
            billing_state = billing.get('state')
            billing_zip = billing.get('zipCode')
        
        data = {
            "po_id": rec.get("poId"),
            "created_time": rec.get("createdTime"),
            "last_modified_time": rec.get("lastModifiedTime"),
            "client_po_num": rec.get("clientPONumber"),
            "client_plain_text": rec.get("clientPlainText"),
            "client_id": client_id,
            "client_name": client_name,  # CUSTOMER NAME FIELD
            "client_code": client_code,
            "total_amount": float(rec.get("totalAmount")) if rec.get("totalAmount") else 0.0,
            "status": rec.get("status"),
            "due_date": rec.get("dueDate"),
            "estimated_start_date": rec.get("estimatedStartDate"),
            "description": rec.get("description"),
            "internal_notes": rec.get("internalNotes"),
            "contact": rec.get("contact"),
            "shipping_address": shipping_addr,
            "shipping_city": shipping_city if shipping else None,
            "shipping_state": shipping_state if shipping else None,
            "shipping_zip": shipping_zip if shipping else None,
            "billing_address": billing_addr,
            "billing_city": billing_city if billing else None,
            "billing_state": billing_state if billing else None,
            "billing_zip": billing_zip if billing else None,
        }
        parsed_records.append(data)
    
    df = pd.DataFrame(parsed_records)
    return df

def load_to_azure(df):
    from azure import load_client_po, connect_to_db, connect_local
    logging.info("Loading data to Azure SQL Database...")
    conn = connect_local()
    load_client_po(df, conn)


if __name__ == "__main__":
    token = get_token()
    if token:
        records = fetch_po(token)
        if records:
            df = parse_data(records)
            
            # Display sample data with customer names
            logging.info(f"\nSample data with customer names:\n{df[['client_po_num', 'client_name', 'total_amount']].head()}")
            
            load_to_azure(df)

KeyError: 'data'